# Fase 7 — eICU Etiqueta L3 (vasopresor o ventilacion <= 60 min)

Construcción de la etiqueta L3 (intervención crítica) para eICU:
- **Vasopresor** en `infusionDrug` con `infusionoffset` entre 0 y 60 minutos.
- **Ventilación mecánica** en `respiratoryCare` con `ventstartoffset` entre 0 y 60 minutos.

La definición es análoga a la de MIMIC-IV-ED, pero la prevalencia resultante (~19%) difiere considerablemente de la de MIMIC (~0,58%) porque los pacientes de eICU ya se encuentran en estado crítico en el momento del ingreso a UCI. Esta diferencia de distribución se documenta como limitación del análisis de validación externa.

In [1]:
import logging, sys
from pathlib import Path

_root = Path.cwd()
while not (_root / 'CLAUDE.md').exists() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

logging.basicConfig(level=logging.INFO, format='%(levelname)s — %(message)s', force=True)
print(f'Raíz: {_root}')

Raíz: C:\Users\cuent\Documents\UAX\TFM\TFM


In [2]:
import pandas as pd

EICU = _root / 'data/eicu-collaborative-research-database-2.0'
OUT  = _root / 'data/processed'

cohort   = pd.read_parquet(OUT / 'eicu_cohort.parquet')
stay_ids = set(cohort['patientunitstayid'].values)
print(f'Cohorte: {len(stay_ids):,} estancias')

Cohorte: 89,594 estancias


## 1. Vasopresores en infusionDrug (offset 0–60 min)

In [3]:
VASOPRESSORS = [
    'norepinephrine', 'noradrenaline', 'levophed',
    'vasopressin', 'pitressin',
    'dopamine',
    'epinephrine', 'adrenaline',
    'phenylephrine', 'neosynephrine',
    'dobutamine',
]

inf = pd.read_csv(
    EICU / 'infusionDrug.csv',
    usecols=['patientunitstayid', 'infusionoffset', 'drugname'],
)
inf = inf[inf['patientunitstayid'].isin(stay_ids)].copy()
inf = inf[(inf['infusionoffset'] >= 0) & (inf['infusionoffset'] <= 60)]

pattern = '|'.join(VASOPRESSORS)
inf['is_vaso'] = inf['drugname'].str.lower().fillna('').str.contains(pattern, regex=True)

print('Farmacos identificados como vasopresores (top 10):')
print(inf[inf['is_vaso']]['drugname'].value_counts().head(10))

vaso_stays = set(inf.loc[inf['is_vaso'], 'patientunitstayid'].unique())
print(f'\nEstancias con vasopresor <=60 min: {len(vaso_stays):,}')

Farmacos identificados como vasopresores (top 10):
drugname
Norepinephrine (mcg/min)       2071
Norepinephrine (ml/hr)         1073
Dopamine (mcg/kg/min)           472
Norepinephrine (mcg/kg/min)     469
Vasopressin (units/min)         301
Dopamine (ml/hr)                231
Epinephrine (mcg/min)           162
Phenylephrine (mcg/min)         147
Norepinephrine ()                86
Phenylephrine (ml/hr)            82
Name: count, dtype: int64

Estancias con vasopresor <=60 min: 3,152


## 2. Ventilación mecánica en respiratoryCare (offset 0–60 min)

`ventstartoffset` indica el minuto desde el ingreso a UCI en que se inicia la ventilación mecánica invasiva. Se aplica el mismo criterio temporal que para vasopresores (ventana 0–60 min).

In [4]:
resp = pd.read_csv(
    EICU / 'respiratoryCare.csv',
    usecols=['patientunitstayid', 'ventstartoffset'],
)
resp = resp[resp['patientunitstayid'].isin(stay_ids)].copy()
resp = resp[(resp['ventstartoffset'] >= 0) & (resp['ventstartoffset'] <= 60)]

vent_stays = set(resp['patientunitstayid'].unique())
print(f'Estancias con ventilación ≤60 min: {len(vent_stays):,}')

Estancias con ventilación ≤60 min: 15,062


## 3. Etiqueta L3 (unión vasopresor OR ventilación)

In [5]:
l3 = cohort[['patientunitstayid']].copy()
l3['L3_vaso'] = l3['patientunitstayid'].isin(vaso_stays).astype(int)
l3['L3_vent'] = l3['patientunitstayid'].isin(vent_stays).astype(int)
l3['L3']      = l3['patientunitstayid'].isin(vaso_stays | vent_stays).astype(int)

print(f'L3 (vasopresor):   {l3["L3_vaso"].mean():.3%}')
print(f'L3 (ventilación):  {l3["L3_vent"].mean():.3%}')
print(f'L3 (combinado):    {l3["L3"].mean():.3%}  ← vs 0.58% en MIMIC')
print(f'L3 positivos:      {l3["L3"].sum():,} / {len(l3):,}')

L3 (vasopresor):   3.518%
L3 (ventilación):  16.811%
L3 (combinado):    19.117%  ← vs 0.58% en MIMIC
L3 positivos:      17,128 / 89,594


## 4. Guardar

Serialización de la etiqueta L3 en `data/processed/eicu_l3.parquet`. Se incluyen solo `patientunitstayid` y `L3` para mantener independencia respecto a otros artefactos del pipeline.

In [6]:
l3[['patientunitstayid', 'L3']].to_parquet(OUT / 'eicu_l3.parquet', index=False)
print(f'Guardado: eicu_l3.parquet')

Guardado: eicu_l3.parquet
